# Part 22: Feature Engineering

Financial Data Science II
Updated March 10, 2026



## Context: Finding Alpha

There are a couple of ways to find alpha.

**Part 1** focuses on using historical data and engineered features to train a model that predicts future returns.

**Part 2** (to be filled later) will discuss factor models (e.g., 3-factor or 5-factor), long/short construction, and how to translate factor exposures into tradeable features.







## Part 1: Historical Data + Feature Engineering

We will:
1. Download US stock data from Yahoo Finance (via `yfinance`).
2. Build price-based features.
3. Add fundamental ratios (P/E, P/B) when available.
4. Define a forward-return target.
5. Run a walk-forward evaluation.



In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor




In [2]:
# Parameters
TICKER = "AAPL"
START = "2016-01-01"
END = "2026-03-01"

px = yf.download(TICKER, start=START, end=END, auto_adjust=False, progress=False)
px = px.rename(columns={"Adj Close": "adj_close"})
px = px.dropna()
px.head()



Price,adj_close,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,,
2016-01-04,23.730949,26.337500,26.342501,25.500000,25.652500,270597600
2016-01-05,23.136265,25.677500,26.462500,25.602501,26.437500,223164000
2016-01-06,22.683496,25.174999,25.592501,24.967501,25.139999,273829600
2016-01-07,21.726149,24.112499,25.032499,24.107500,24.670000,324377600
2016-01-08,21.841036,24.240000,24.777500,24.190001,24.637501,283192000


In [3]:
# Price-based features
feat = pd.DataFrame(index=px.index)
feat["ret_1d"] = px["adj_close"].pct_change()
feat["log_ret_1d"] = np.log(px["adj_close"] / px["adj_close"].shift(1))

for w in [5, 21, 63, 126]:
    feat[f"mom_{w}d"] = px["adj_close"].pct_change(w)
    feat[f"vol_{w}d"] = feat["ret_1d"].rolling(w).std()

rolling_max = px["adj_close"].cummax()
feat["drawdown"] = px["adj_close"] / rolling_max - 1

feat = feat.dropna()
feat.head()



,ret_1d,log_ret_1d,mom_5d,vol_5d,mom_21d,vol_21d,mom_63d,vol_63d,mom_126d,vol_126d,drawdown
Date,,,,,,,,,,,
2016-07-05,-0.009386,-0.009430,0.032051,0.010179,-0.029922,0.011551,-0.129694,0.014660,-0.087927,0.016891,-0.147472
2016-07-06,0.005685,0.005669,0.020729,0.008373,-0.031430,0.011498,-0.133817,0.014607,-0.059165,0.016756,-0.142626
2016-07-07,0.004291,0.004282,0.016313,0.008004,-0.031203,0.011504,-0.110705,0.014409,-0.036267,0.016672,-0.138946
2016-07-08,0.007713,0.007684,0.011297,0.006742,-0.022842,0.011676,-0.104835,0.014455,0.013960,0.016253,-0.132305
2016-07-11,0.003103,0.003098,0.011367,0.006744,-0.026794,0.011566,-0.105022,0.014453,0.011756,0.016248,-0.129612


### Step B: Fundamentals (P/E, P/B)
We keep this short and readable. All logic is in small helper functions.


In [4]:
def get_table(ticker, attr):
    obj = getattr(ticker, attr, None)
    if isinstance(obj, pd.DataFrame):
        return obj
    return pd.DataFrame()

def get_shares(ticker):
    shares = None
    try:
        shares = ticker.get_shares_full()
    except Exception:
        try:
            shares = ticker.get_shares()
        except Exception:
            shares = None
    if isinstance(shares, pd.DataFrame) and shares.shape[1] > 0:
        shares = shares.iloc[:, 0]
    if shares is not None and hasattr(shares, 'index'):
        shares = shares[~shares.index.duplicated(keep='last')]
        try:
            shares.index = pd.to_datetime(shares.index).tz_localize(None)
        except Exception:
            shares.index = pd.to_datetime(shares.index, errors='coerce')
    return shares

def build_eps_bvps(income_q, balance_q, shares, index):
    fund = pd.DataFrame(index=index)
    if not income_q.empty and shares is not None and len(shares) > 0:
        iq = income_q.T
        iq.index = pd.to_datetime(iq.index)
        col = next((c for c in iq.columns if 'Net Income' in c), None)
        if col:
            eps = iq[col] / shares.reindex(iq.index, method='nearest')
            if isinstance(eps, pd.DataFrame) and eps.shape[1] > 0:
                eps = eps.iloc[:, 0]
            fund['eps_q'] = eps.replace([np.inf, -np.inf], np.nan).reindex(index, method='ffill')
    if not balance_q.empty and shares is not None and len(shares) > 0:
        bq = balance_q.T
        bq.index = pd.to_datetime(bq.index)
        col = next((c for c in bq.columns if 'Total Stockholder Equity' in c or 'Total Equity' in c), None)
        if col:
            bvps = bq[col] / shares.reindex(bq.index, method='nearest')
            if isinstance(bvps, pd.DataFrame) and bvps.shape[1] > 0:
                bvps = bvps.iloc[:, 0]
            fund['bvps_q'] = bvps.replace([np.inf, -np.inf], np.nan).reindex(index, method='ffill')
    return fund

def add_pe_pb(fund, price):
    fund = fund.loc[:, ~fund.columns.duplicated()]
    price = price.reindex(fund.index)

    def _to_series(x):
        if isinstance(x, pd.DataFrame):
            if x.shape[1] > 0:
                x = x.iloc[:, 0]
        return x

    pe = None
    if 'eps_q' in fund.columns:
        eps = _to_series(fund['eps_q'])
        pe = _to_series(price / eps)

    pb = None
    if 'bvps_q' in fund.columns:
        bvps = _to_series(fund['bvps_q'])
        pb = _to_series(price / bvps)

    fund['pe'] = pe
    fund['pb'] = pb
    return fund




In [5]:
tk = yf.Ticker(TICKER)
income_q = get_table(tk, 'quarterly_income_stmt')
if income_q.empty:
    income_q = get_table(tk, 'income_stmt')

balance_q = get_table(tk, 'quarterly_balance_sheet')
if balance_q.empty:
    balance_q = get_table(tk, 'balance_sheet')

shares = get_shares(tk)
fund = build_eps_bvps(income_q, balance_q, shares, px.index)
fund = add_pe_pb(fund, px['adj_close'])

features = feat.join(fund[['pe', 'pb']], how='left')
if features.dropna().empty:
    features = feat.copy()
else:
    features = features.dropna()

features.head()


,ret_1d,log_ret_1d,mom_5d,vol_5d,mom_21d,vol_21d,mom_63d,vol_63d,mom_126d,vol_126d,drawdown
Date,,,,,,,,,,,
2016-07-05,-0.009386,-0.009430,0.032051,0.010179,-0.029922,0.011551,-0.129694,0.014660,-0.087927,0.016891,-0.147472
2016-07-06,0.005685,0.005669,0.020729,0.008373,-0.031430,0.011498,-0.133817,0.014607,-0.059165,0.016756,-0.142626
2016-07-07,0.004291,0.004282,0.016313,0.008004,-0.031203,0.011504,-0.110705,0.014409,-0.036267,0.016672,-0.138946
2016-07-08,0.007713,0.007684,0.011297,0.006742,-0.022842,0.011676,-0.104835,0.014455,0.013960,0.016253,-0.132305
2016-07-11,0.003103,0.003098,0.011367,0.006744,-0.026794,0.011566,-0.105022,0.014453,0.011756,0.016248,-0.129612


In [6]:
# Fundamental ratios (best-effort from financial statements)

def _get_quarterly_series(ticker: yf.Ticker, attr: str) -> pd.DataFrame:
    obj = getattr(ticker, attr, None)
    if obj is None:
        return pd.DataFrame()
    if isinstance(obj, pd.DataFrame):
        return obj
    return pd.DataFrame()


tk = yf.Ticker(TICKER)

income_q = _get_quarterly_series(tk, "quarterly_income_stmt")
if income_q.empty:
    income_q = _get_quarterly_series(tk, "income_stmt")

balance_q = _get_quarterly_series(tk, "quarterly_balance_sheet")
if balance_q.empty:
    balance_q = _get_quarterly_series(tk, "balance_sheet")

shares = None
try:
    shares = tk.get_shares_full()
except Exception:
    try:
        shares = tk.get_shares()
    except Exception:
        shares = None

# If shares is a DataFrame, take the first column as a Series
if isinstance(shares, pd.DataFrame) and shares.shape[1] > 0:
    shares = shares.iloc[:, 0]

# Deduplicate share index (some tickers return multiple entries per date)
if shares is not None and hasattr(shares, 'index'):
    shares = shares[~shares.index.duplicated(keep='last')]

# Align share index timezone to naive for reindexing
if shares is not None and hasattr(shares, 'index'):
    try:
        shares.index = pd.to_datetime(shares.index).tz_localize(None)
    except Exception:
        shares.index = pd.to_datetime(shares.index, errors='coerce')

fund = pd.DataFrame(index=px.index)

if not income_q.empty and shares is not None and len(shares) > 0:
    income_q = income_q.T
    income_q.index = pd.to_datetime(income_q.index)
    net_income_col = None
    for c in income_q.columns:
        if "Net Income" in c:
            net_income_col = c
            break
    if net_income_col:
        eps_q = income_q[net_income_col] / shares.reindex(income_q.index, method="nearest")
        if isinstance(eps_q, pd.DataFrame) and eps_q.shape[1] > 0:
            eps_q = eps_q.iloc[:, 0]
        eps_q = eps_q.replace([np.inf, -np.inf], np.nan)
        fund["eps_q"] = eps_q.reindex(fund.index, method="ffill")

if not balance_q.empty and shares is not None and len(shares) > 0:
    balance_q = balance_q.T
    balance_q.index = pd.to_datetime(balance_q.index)
    equity_col = None
    for c in balance_q.columns:
        if "Total Stockholder Equity" in c or "Total Equity" in c:
            equity_col = c
            break
    if equity_col:
        bvps_q = balance_q[equity_col] / shares.reindex(balance_q.index, method="nearest")
        if isinstance(bvps_q, pd.DataFrame) and bvps_q.shape[1] > 0:
            bvps_q = bvps_q.iloc[:, 0]
        bvps_q = bvps_q.replace([np.inf, -np.inf], np.nan)
        fund["bvps_q"] = bvps_q.reindex(fund.index, method="ffill")

# Build ratios robustly (handle missing/extra columns)
fund = fund.loc[:, ~fund.columns.duplicated()]
price_series = px["adj_close"].reindex(fund.index)

pe_series = None
if "eps_q" in fund.columns:
    eps = fund["eps_q"]
    if isinstance(eps, pd.DataFrame) and eps.shape[1] > 0:
        eps = eps.iloc[:, 0]
    pe_series = price_series / eps
    pe_series = pe_series.squeeze()
    if isinstance(pe_series, pd.DataFrame) and pe_series.shape[1] > 0:
        pe_series = pe_series.iloc[:, 0]

pb_series = None
if "bvps_q" in fund.columns:
    bvps = fund["bvps_q"]
    if isinstance(bvps, pd.DataFrame) and bvps.shape[1] > 0:
        bvps = bvps.iloc[:, 0]
    pb_series = price_series / bvps
    pb_series = pb_series.squeeze()
    if isinstance(pb_series, pd.DataFrame) and pb_series.shape[1] > 0:
        pb_series = pb_series.iloc[:, 0]

fund["pe"] = pe_series
fund["pb"] = pb_series

features = feat.join(fund[["pe", "pb"]], how="left")
if features.dropna().empty:
    # If fundamentals unavailable, fall back to price-only features
    features = feat.copy()
else:
    features = features.dropna()
features.head()









,ret_1d,log_ret_1d,mom_5d,vol_5d,mom_21d,vol_21d,mom_63d,vol_63d,mom_126d,vol_126d,drawdown
Date,,,,,,,,,,,
2016-07-05,-0.009386,-0.009430,0.032051,0.010179,-0.029922,0.011551,-0.129694,0.014660,-0.087927,0.016891,-0.147472
2016-07-06,0.005685,0.005669,0.020729,0.008373,-0.031430,0.011498,-0.133817,0.014607,-0.059165,0.016756,-0.142626
2016-07-07,0.004291,0.004282,0.016313,0.008004,-0.031203,0.011504,-0.110705,0.014409,-0.036267,0.016672,-0.138946
2016-07-08,0.007713,0.007684,0.011297,0.006742,-0.022842,0.011676,-0.104835,0.014455,0.013960,0.016253,-0.132305
2016-07-11,0.003103,0.003098,0.011367,0.006744,-0.026794,0.011566,-0.105022,0.014453,0.011756,0.016248,-0.129612


In [7]:
# Target: 5-day forward return
TARGET_H = 5
features["target_fwd_ret"] = px["adj_close"].reindex(features.index).shift(-TARGET_H) / px["adj_close"].reindex(features.index) - 1
features = features.dropna()

X = features.drop(columns=["target_fwd_ret"])
y = features["target_fwd_ret"]

X.head(), y.head()



(              ret_1d  log_ret_1d    mom_5d    vol_5d   mom_21d   vol_21d  \
 Date                                                                       
 2016-07-05 -0.009386   -0.009430  0.032051  0.010179 -0.029922  0.011551   
 2016-07-06  0.005685    0.005669  0.020729  0.008373 -0.031430  0.011498   
 2016-07-07  0.004291    0.004282  0.016313  0.008004 -0.031203  0.011504   
 2016-07-08  0.007713    0.007684  0.011297  0.006742 -0.022842  0.011676   
 2016-07-11  0.003103    0.003098  0.011367  0.006744 -0.026794  0.011566   
 
              mom_63d   vol_63d  mom_126d  vol_126d  drawdown  
 Date                                                          
 2016-07-05 -0.129694  0.014660 -0.087927  0.016891 -0.147472  
 2016-07-06 -0.133817  0.014607 -0.059165  0.016756 -0.142626  
 2016-07-07 -0.110705  0.014409 -0.036267  0.016672 -0.138946  
 2016-07-08 -0.104835  0.014455  0.013960  0.016253 -0.132305  
 2016-07-11 -0.105022  0.014453  0.011756  0.016248 -0.129612  ,
 Date
 201

### Important: Shift the target (avoid lookahead bias)

If you train with `y` (same-day return) instead of a shifted `y_shift_one` or `target_fwd_ret`, your model will look **much better** but it is **wrong** (it leaks future info).

Bad (leakage, looks great but invalid):
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
```

Correct (shift target forward in time):
```python
X_train, X_test, y_train, y_test = train_test_split(X, y_shift_one, test_size=0.2, shuffle=False)
```

Better (time-series split):
```python
tscv = TimeSeriesSplit(n_splits=5)
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
```



In [8]:
# Walk-forward evaluation
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5,
)

splitter = TimeSeriesSplit(n_splits=5)
mses = []

for train_idx, test_idx in splitter.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mses.append(mean_squared_error(y_test, pred))

print("MSE mean:", float(np.mean(mses)))
print("MSE std:", float(np.std(mses)))



MSE mean: 0.0019457563453682432
MSE std: 0.0004015440165965414


## Model Comparison (Standard Metrics)

We compare multiple models using the same metrics:
- RMSE
- MAE
- R2
- Sign Accuracy (directional hit rate)



In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Optional: XGBoost if installed
try:
    import xgboost as xgb
    has_xgb = True
except Exception:
    has_xgb = False

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1e-3, max_iter=5000),
    "RandomForest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=5
    ),
}

if has_xgb:
    models["XGBoost"] = xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
    )

rows = []
for name, model in models.items():
    model.fit(X_train, y_train.values.ravel())
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    sign_acc = (np.sign(pred) == np.sign(y_test.values)).mean()
    rows.append([name, rmse, mae, r2, sign_acc])

results = pd.DataFrame(rows, columns=["model", "rmse", "mae", "r2", "sign_accuracy"])
results.sort_values("rmse").reset_index(drop=True)






/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/harold/miniconda3

,model,rmse,mae,r2,sign_accuracy
0,Lasso,0.041345,0.030944,-0.000223,0.554639
1,Ridge,0.041360,0.030875,-0.000933,0.556701
2,Linear,0.041862,0.031277,-0.025377,0.536082
3,RandomForest,0.043604,0.032585,-0.112513,0.554639
4,XGBoost,0.044482,0.033418,-0.157791,0.536082


## Part 1B: LOBSTER Microstructure Example

We will:
1. Load the LOBSTER message + orderbook files.
2. Build microstructure features (spread, imbalance, depth).
3. Add OFI (order flow imbalance) as a feature.
4. Predict 1-step mid-price direction.
5. Compare Logistic Regression vs XGBoost (if available).



In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

# Optional: XGBoost
try:
    import xgboost as xgb
    has_xgb = True
except Exception:
    has_xgb = False



In [2]:
lob_base = Path('/Users/harold/4. RA work/FACTOR_TRAINING_CHI/lobster_samples/AAPL_2012-06-21_5')
msg_path = lob_base / 'AAPL_2012-06-21_34200000_57600000_message_5.csv'
ob_path = lob_base / 'AAPL_2012-06-21_34200000_57600000_orderbook_5.csv'

msg = pd.read_csv(msg_path, header=None,
                  names=['time','type','order_id','size','price','direction'])
ob = pd.read_csv(ob_path, header=None)

levels = 5
cols = []
for i in range(1, levels+1):
    cols += [f'ask_price_{i}', f'ask_size_{i}', f'bid_price_{i}', f'bid_size_{i}']
ob.columns = cols

msg.head()




,time,type,order_id,size,price,direction
0,34200.004241,1,16113575,18,5853300,1
1,34200.004261,1,16113584,18,5853200,1
2,34200.004447,1,16113594,18,5853100,1
3,34200.025552,1,16120456,18,5859100,-1
4,34200.025580,1,16120480,18,5859200,-1


In [3]:
ob.head()

,ask_price_1,ask_size_1,bid_price_1,bid_size_1,ask_price_2,ask_size_2,bid_price_2,bid_size_2,ask_price_3,ask_size_3,bid_price_3,bid_size_3,ask_price_4,ask_size_4,bid_price_4,bid_size_4,ask_price_5,ask_size_5,bid_price_5,bid_size_5
0,5859400,200,5853300,18,5859800,200,5853000,150,5861000,200,5851000,5,5868900,300,5850100,89,5869500,50,5849700,5
1,5859400,200,5853300,18,5859800,200,5853200,18,5861000,200,5853000,150,5868900,300,5851000,5,5869500,50,5850100,89
2,5859400,200,5853300,18,5859800,200,5853200,18,5861000,200,5853100,18,5868900,300,5853000,150,5869500,50,5851000,5
3,5859100,18,5853300,18,5859400,200,5853200,18,5859800,200,5853100,18,5861000,200,5853000,150,5868900,300,5851000,5
4,5859100,18,5853300,18,5859200,18,5853200,18,5859400,200,5853100,18,5859800,200,5853000,150,5861000,200,5851000,5


In [4]:
# Basic LOB features
feat_lob = pd.DataFrame(index=msg.index)
feat_lob['time'] = msg['time']
feat_lob['mid'] = (ob['ask_price_1'] + ob['bid_price_1']) / 2
feat_lob['spread'] = ob['ask_price_1'] - ob['bid_price_1']
feat_lob['imbalance_l1'] = (ob['bid_size_1'] - ob['ask_size_1']) / (
    ob['bid_size_1'] + ob['ask_size_1']
)

# Depth imbalance across levels
bid_depth = sum(ob[f'bid_size_{i}'] for i in range(1, levels+1))
ask_depth = sum(ob[f'ask_size_{i}'] for i in range(1, levels+1))
feat_lob['imbalance_l5'] = (bid_depth - ask_depth) / (bid_depth + ask_depth)

feat_lob.head()

,time,mid,spread,imbalance_l1,imbalance_l5
0,34200.004241,5856350.0,6100,-0.834862,-0.561216
1,34200.004261,5856350.0,6100,-0.834862,-0.544715
2,34200.004447,5856350.0,6100,-0.834862,-0.639344
3,34200.025552,5856200.0,5800,0.000000,-0.629104
4,34200.025580,5856200.0,5800,0.000000,-0.505325


### Time Column (Seconds After Midnight)
LOBSTER time is in seconds after midnight. Convert to a readable time for sanity checks.


In [13]:
feat_lob['time_str'] = pd.to_timedelta(feat_lob['time'], unit='s')
feat_lob[['time','time_str']].head()


,time,time_str
0,34200.004241,0 days 09:30:00.004241176
1,34200.004261,0 days 09:30:00.004260640
2,34200.004447,0 days 09:30:00.004447484
3,34200.025552,0 days 09:30:00.025551909
4,34200.025580,0 days 09:30:00.025579546


### Resample to Fixed Intervals (1s Bars)
We use `floor` on timestamps to avoid looking ahead. We aggregate with `last` within each second.


In [14]:
# Build a combined frame for resampling
lob = feat_lob[['time','mid','spread','imbalance_l1','imbalance_l5']].copy()
lob['time_sec'] = lob['time'].astype(float)
lob['time_floor'] = np.floor(lob['time_sec']).astype(int)

# Aggregate to 1-second bars using the last observation in each second
bars_1s = (
    lob.sort_values('time_sec')
       .groupby('time_floor', as_index=False)
       .last()
)

# Convert to readable time
bars_1s['time_str'] = pd.to_timedelta(bars_1s['time_floor'], unit='s')
bars_1s.head()


,time_floor,time,mid,spread,imbalance_l1,imbalance_l5,time_sec,time_str
0,34200,34200.918221,5858050.0,1300,0.000000,-0.673647,34200.918221,0 days 09:30:00
1,34201,34201.840852,5856200.0,3000,0.000000,0.111111,34201.840852,0 days 09:30:01
2,34202,34202.756909,5855750.0,2100,0.694915,0.333333,34202.756909,0 days 09:30:02
3,34203,34203.954770,5855800.0,2600,-0.694915,-0.166861,34203.954770,0 days 09:30:03
4,34204,34204.894471,5855900.0,1800,-0.960784,-0.619938,34204.894471,0 days 09:30:04


### Resample Full L1 Quotes (Bid/Ask + Sizes)
We resample raw L1 quotes using last observation per second, then recompute mid/spread/imbalance.


In [15]:
# Build a level-1 quote frame
lob_l1 = pd.DataFrame({
    'time': msg['time'],
    'bid_1': ob['bid_price_1'],
    'bid_size_1': ob['bid_size_1'],
    'ask_1': ob['ask_price_1'],
    'ask_size_1': ob['ask_size_1'],
})
lob_l1['time_sec'] = lob_l1['time'].astype(float)
lob_l1['time_floor'] = np.floor(lob_l1['time_sec']).astype(int)

# Last quote per second
l1_1s = (
    lob_l1.sort_values('time_sec')
          .groupby('time_floor', as_index=False)
          .last()
)

# Recompute features on bars
l1_1s['mid'] = (l1_1s['ask_1'] + l1_1s['bid_1']) / 2
l1_1s['spread'] = l1_1s['ask_1'] - l1_1s['bid_1']
l1_1s['imbalance_l1'] = (l1_1s['bid_size_1'] - l1_1s['ask_size_1']) / (
    l1_1s['bid_size_1'] + l1_1s['ask_size_1']
)
l1_1s['time_str'] = pd.to_timedelta(l1_1s['time_floor'], unit='s')

l1_1s.head()


,time_floor,time,bid_1,bid_size_1,ask_1,ask_size_1,time_sec,mid,spread,imbalance_l1,time_str
0,34200,34200.918221,5857400,100,5858700,100,34200.918221,5858050.0,1300,0.000000,0 days 09:30:00
1,34201,34201.840852,5854700,18,5857700,18,34201.840852,5856200.0,3000,0.000000,0 days 09:30:01
2,34202,34202.756909,5854700,100,5856800,18,34202.756909,5855750.0,2100,0.694915,0 days 09:30:02
3,34203,34203.954770,5854500,18,5857100,100,34203.954770,5855800.0,2600,-0.694915,0 days 09:30:03
4,34204,34204.894471,5855000,18,5856800,900,34204.894471,5855900.0,1800,-0.960784,0 days 09:30:04


In [16]:
l1_1s

,time_floor,time,bid_1,bid_size_1,ask_1,ask_size_1,time_sec,mid,spread,imbalance_l1,time_str
0,34200,34200.918221,5857400,100,5858700,100,34200.918221,5858050.0,1300,0.000000,0 days 09:30:00
1,34201,34201.840852,5854700,18,5857700,18,34201.840852,5856200.0,3000,0.000000,0 days 09:30:01
2,34202,34202.756909,5854700,100,5856800,18,34202.756909,5855750.0,2100,0.694915,0 days 09:30:02
3,34203,34203.954770,5854500,18,5857100,100,34203.954770,5855800.0,2600,-0.694915,0 days 09:30:03
4,34204,34204.894471,5855000,18,5856800,900,34204.894471,5855900.0,1800,-0.960784,0 days 09:30:04
...,...,...,...,...,...,...,...,...,...,...,...
19885,57595,57595.987547,5775500,656,5776400,943,57595.987547,5775950.0,900,-0.179487,0 days 15:59:55
19886,57596,57596.798348,5776000,400,5776400,716,57596.798348,5776200.0,400,-0.283154,0 days 15:59:56
19887,57597,57597.812997,5776000,500,5776700,1300,57597.812997,5776350.0,700,-0.444444,0 days 15:59:57
19888,57598,57598.937053,5775900,150,5776700,1062,57598.937053,5776300.0,800,-0.752475,0 days 15:59:58


In [17]:
# OFI (order flow imbalance)
# Simplified: sign(type/dir) * size
# Type 1/2/3 are order submissions/cancels/deletions, 4/5 executions.
# We use direction and size to build a signed flow measure.

msg_flow = msg.copy()
msg_flow['signed_size'] = msg_flow['direction'] * msg_flow['size']

# Aggregate over short window to reduce noise
window = 20
feat_lob['ofi_20'] = msg_flow['signed_size'].rolling(window).sum()

feat_lob.head()



,time,mid,spread,imbalance_l1,imbalance_l5,time_str,ofi_20
0,34200.004241,5856350.0,6100,-0.834862,-0.561216,0 days 09:30:00.004241176,NaN
1,34200.004261,5856350.0,6100,-0.834862,-0.544715,0 days 09:30:00.004260640,NaN
2,34200.004447,5856350.0,6100,-0.834862,-0.639344,0 days 09:30:00.004447484,NaN
3,34200.025552,5856200.0,5800,0.000000,-0.629104,0 days 09:30:00.025551909,NaN
4,34200.025580,5856200.0,5800,0.000000,-0.505325,0 days 09:30:00.025579546,NaN


In [18]:
# Target: 1-step mid-price direction
feat_lob['mid_ret_1'] = feat_lob['mid'].shift(-1) / feat_lob['mid'] - 1
feat_lob['y_up'] = (feat_lob['mid_ret_1'] > 0).astype(int)

# Drop last row and NaNs
feat_lob = feat_lob.dropna()

X = feat_lob[['spread','imbalance_l1','imbalance_l5','ofi_20']]
y = feat_lob['y_up']
X.head(), y.head()



(    spread  imbalance_l1  imbalance_l5  ofi_20
 19     100     -0.333333     -0.650485   -87.0
 20     100     -0.333333     -0.680791  -100.0
 21     100     -0.333333     -0.683916  -125.0
 22     100     -0.333333     -0.679219  -123.0
 23     100     -0.333333     -0.687924  -125.0,
 19    0
 20    0
 21    0
 22    0
 23    0
 Name: y_up, dtype: int64)

In [19]:
# Time-series split evaluation
splitter = TimeSeriesSplit(n_splits=5)

results = []
for train_idx, test_idx in splitter.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Logistic
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    p_lr = lr.predict_proba(X_test)[:, 1]
    pred_lr = (p_lr > 0.5).astype(int)
    acc_lr = accuracy_score(y_test, pred_lr)
    auc_lr = roc_auc_score(y_test, p_lr)
    results.append(['Logistic', acc_lr, auc_lr])

    if has_xgb:
        xgb_model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric='logloss'
        )
        xgb_model.fit(X_train, y_train)
        p_xgb = xgb_model.predict_proba(X_test)[:, 1]
        pred_xgb = (p_xgb > 0.5).astype(int)
        acc_xgb = accuracy_score(y_test, pred_xgb)
        auc_xgb = roc_auc_score(y_test, p_xgb)
        results.append(['XGBoost', acc_xgb, auc_xgb])

res = pd.DataFrame(results, columns=['model','accuracy','auc'])
res.groupby('model').mean().reset_index()



/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul

/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul

/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul

/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul

/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/harold/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul

,model,accuracy,auc
0,Logistic,0.899712,0.554644
1,XGBoost,0.899596,0.580394
